# Data prep: sql-create-context

Pull the dataset from HF, clean it up a bit, and write out a fixed train/eval split so the training and eval notebooks always see the same data.

Runs on CPU, no GPU needed.

In [1]:
from datasets import load_dataset

ds = load_dataset("b-mc2/sql-create-context", split="train")
ds

README.md: 0.00B [00:00, ?B/s]

E:\ir-infotech-ml-assignment\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in E:\DevStorage\.cache\huggingface\hub\datasets--b-mc2--sql-create-context. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Dataset({
    features: ['answer', 'question', 'context'],
    num_rows: 78577
})

In [2]:
# what a row looks like
ds[0]

{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56',
 'question': 'How many heads of the departments are older than 56 ?',
 'context': 'CREATE TABLE head (age INTEGER)'}

In [3]:
import pandas as pd

df = ds.to_pandas()
df["q_len"] = df["question"].str.len()
df["ctx_len"] = df["context"].str.len()
df["a_len"] = df["answer"].str.len()
df[["q_len", "ctx_len", "a_len"]].describe().round(1)

,q_len,ctx_len,a_len
count,78577.0,78577.0,78577.0
mean,60.7,71.6,76.3
std,23.2,19.6,24.5
min,12.0,27.0,18.0
25%,43.0,60.0,62.0
50%,57.0,68.0,71.0
75%,73.0,77.0,85.0
max,244.0,489.0,557.0


Lengths are mostly short, which is good for a 512 token budget on the T4. There are some duplicated questions in this dataset though (same question paired with near-identical schemas), so dedupe first to avoid train/eval leakage.

In [4]:
before = len(df)
df = df.drop_duplicates(subset=["question"]).reset_index(drop=True)
print(f"dropped {before - len(df)} duplicate questions, {len(df)} rows left")

dropped 266 duplicate questions, 78311 rows left


In [5]:
# rough char cutoffs so full prompts comfortably fit in 512 tokens later
df = df[(df["q_len"] < 300) & (df["ctx_len"] < 500) & (df["a_len"] < 300)].reset_index(drop=True)
len(df)

78243

In [6]:
import os

# fixed seed so this split is reproducible
sample = df.sample(n=8500, random_state=42).reset_index(drop=True)
cols = ["question", "context", "answer"]
train_df = sample.loc[:7999, cols]
eval_df = sample.loc[8000:, cols]

os.makedirs("../data", exist_ok=True)
train_df.to_json("../data/train.jsonl", orient="records", lines=True)
eval_df.to_json("../data/eval.jsonl", orient="records", lines=True)
print(len(train_df), "train /", len(eval_df), "eval")

8000 train / 500 eval


8000 train / 500 eval. Both files are committed to the repo so training and eval never drift out of sync with each other.